### Modélisation Machine Learning des demandes d’asile

### Objectif

Construire un modèle de **classification binaire** capable de prédire si un segment de demandes d’asile connaîtra une hausse supérieure à **15 % à T+1**.

La cible est :

`hausse_critique_demandes_t1`

- `1` : hausse > 15 % à T+1 ;
- `0` : hausse ≤ 15 % à T+1.

### Protocole temporel retenu

| Ensemble | Années T | Années prédites T+1 | Rôle |
|---|---:|---:|---|
| Train | 2000–2021 | 2001–2022 | apprentissage |
| Validation | 2022–2023 | 2023–2024 | comparaison et réglage |
| Test final | 2024 | 2025 | évaluation finale |

> **Règle : le Test 2024 → 2025 reste verrouillé jusqu’au choix final du modèle.**


### 1. Imports et configuration

Cette première version du notebook va jusqu’au **split chronologique** et à la **baseline naïve**.  
Le preprocessing et les modèles seront ajoutés ensuite, étape par étape.


In [16]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

### 2. Chargement du Feature Set

Le fichier provient du notebook de Feature Engineering.

Il contient uniquement les lignes pour lesquelles la cible T+1 est observable.


In [17]:
DATA_PATH = Path("../data/feature_engineering_outputs/dataset_ml_features.csv")

assert DATA_PATH.exists(), (
    "Le fichier Feature Engineering n'existe pas. "
    "Exécuter d'abord 02_Feature_Engineering_Demandes_Asile.ipynb."
)

df_ml = pd.read_csv(DATA_PATH)

TARGET = "hausse_critique_demandes_t1"

print(f"Dimensions : {df_ml.shape[0]:,} lignes × {df_ml.shape[1]} colonnes")
print(f"Période disponible : {df_ml['year'].min()}–{df_ml['year'].max()}")
display(df_ml.head())

Dimensions : 80,032 lignes × 34 colonnes
Période disponible : 2000–2024


,coo_id,coa_id,procedure_type,app_type,dec_level,app_pc,origin_region,asylum_region,year,applied,log_applied,applied_lag1,applied_lag2,absolute_change_past_1,absolute_change_past_2,growth_past_1,growth_past_2,two_consecutive_increases,applied_mean_3y,applied_std_3y,origin_applied_total_t,asylum_applied_total_t,log_origin_applied_total_t,log_asylum_applied_total_t,segment_share_origin_t,segment_share_asylum_t,origin_applied_total_lag1,idmc_total_origin_lag1,log_idmc_total_origin_lag1,idmc_pressure_ratio_lag1,decisions_total_lag1,protection_rate_lag1,rejection_rate_lag1,hausse_critique_demandes_t1
0,2,11,G,A,AR,C,SOUTHERN ASIA,AUSTRALIA-NEW ZEALAND,2006,14,2.7081,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,87,3030,4.4773,8.0166,0.1609,0.0046,211.0000,NaN,NaN,NaN,NaN,NaN,NaN,0
1,3,11,G,A,AR,C,SOUTHERN EUROPE,AUSTRALIA-NEW ZEALAND,2006,21,3.0910,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,303,3030,5.7170,8.0166,0.0693,0.0069,336.0000,NaN,NaN,NaN,NaN,NaN,NaN,0
2,4,11,G,A,AR,C,NORTHERN AFRICA,AUSTRALIA-NEW ZEALAND,2006,5,1.7918,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,22,3030,3.1355,8.0166,0.2273,0.0017,16.0000,NaN,NaN,NaN,NaN,NaN,NaN,0
3,8,11,G,A,AR,C,NORTHERN AFRICA,AUSTRALIA-NEW ZEALAND,2006,38,3.6636,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,377,3030,5.9349,8.0166,0.1008,0.0125,291.0000,NaN,NaN,NaN,NaN,NaN,NaN,0
4,14,11,G,A,AR,C,WESTERN ASIA,AUSTRALIA-NEW ZEALAND,2006,11,2.4849,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11,3030,2.4849,8.0166,1.0000,0.0036,5.0000,NaN,NaN,NaN,NaN,NaN,NaN,0


### 3. Contrôles préalables

Avant de séparer les données, on vérifie :

- que la cible existe ;
- qu’elle ne contient que `0` et `1` ;
- qu’aucune variable explicitement future n’est utilisée comme feature.


In [18]:
assert TARGET in df_ml.columns, f"Cible absente : {TARGET}"

target_values = set(df_ml[TARGET].dropna().unique())
assert target_values.issubset({0, 1}), f"Valeurs inattendues dans la cible : {target_values}"

forbidden_features = {
    "year_t1",
    "applied_t1",
    "growth_t1",
    TARGET
}

feature_cols = [c for c in df_ml.columns if c != TARGET]
leakage_found = forbidden_features.intersection(feature_cols)

assert not leakage_found, f"Leakage explicite détecté : {leakage_found}"

print("✓ Cible valide")
print("✓ Aucun champ futur explicite dans les features")

✓ Cible valide
✓ Aucun champ futur explicite dans les features


### 4. Split chronologique Train / Validation / Test

Contrairement à un split aléatoire, le split chronologique respecte le sens réel de la prédiction :

**passé → apprentissage → futur → évaluation**

Le modèle ne doit jamais apprendre sur une année postérieure à celle qu’il cherche à prédire.


In [19]:
train_mask = df_ml["year"] <= 2021
val_mask   = df_ml["year"].between(2022, 2023)
test_mask  = df_ml["year"] == 2024

train_df = df_ml.loc[train_mask].copy()
val_df   = df_ml.loc[val_mask].copy()
test_df  = df_ml.loc[test_mask].copy()

assert len(train_df) + len(val_df) + len(test_df) == len(df_ml)
assert train_df["year"].max() < val_df["year"].min()
assert val_df["year"].max() < test_df["year"].min()

split_summary = pd.DataFrame({
    "ensemble": ["Train", "Validation", "Test final"],
    "annees_T": [
        f"{train_df['year'].min()}–{train_df['year'].max()}",
        f"{val_df['year'].min()}–{val_df['year'].max()}",
        str(test_df['year'].min())
    ],
    "annees_predites_T+1": [
        f"{train_df['year'].min()+1}–{train_df['year'].max()+1}",
        f"{val_df['year'].min()+1}–{val_df['year'].max()+1}",
        str(test_df['year'].min()+1)
    ],
    "n_lignes": [len(train_df), len(val_df), len(test_df)]
})

display(split_summary)

,ensemble,annees_T,annees_predites_T+1,n_lignes
0,Train,2000–2021,2001–2022,66237
1,Validation,2022–2023,2023–2024,9342
2,Test final,2024,2025,4453


### Lecture du split

- **Train** : le modèle apprend les relations historiques.
- **Validation** : on compare les modèles et on règle les hyperparamètres / seuils.
- **Test final** : il ne sert ni à apprendre ni à choisir le modèle. Il sera ouvert uniquement à la fin.

Pour notre cas, une ligne `year = 2024` sert à prédire ce qui se passe en **2025**.


### 5. Construction de X et y

`X` = variables explicatives.  
`y` = cible à prédire.

Les identifiants pays (`coo_id`, `coa_id`) resteront des **variables catégorielles**, même s’ils sont stockés sous forme numérique.


In [20]:
X_train = train_df.drop(columns=[TARGET])
y_train = train_df[TARGET].astype(int)

X_val = val_df.drop(columns=[TARGET])
y_val = val_df[TARGET].astype(int)

# Le test est préparé mais ses performances ne seront pas consultées maintenant.
X_test = test_df.drop(columns=[TARGET])
y_test = test_df[TARGET].astype(int)

print("Train      :", X_train.shape, y_train.shape)
print("Validation :", X_val.shape, y_val.shape)
print("Test final :", X_test.shape, y_test.shape)

Train      : (66237, 33) (66237,)
Validation : (9342, 33) (9342,)
Test final : (4453, 33) (4453,)


### 6. Distribution de la cible sur Train et Validation

Avant tout modèle, on mesure le déséquilibre des classes sur les ensembles utilisés pour le développement.

Le Test final reste volontairement hors de cette analyse.


In [21]:
def target_distribution(y, label):
    counts = y.value_counts().sort_index()
    result = pd.DataFrame({
        "classe": [0, 1],
        "effectif": [int(counts.get(0, 0)), int(counts.get(1, 0))],
    })
    result["pourcentage"] = 100 * result["effectif"] / len(y)
    result.insert(0, "ensemble", label)
    return result

distribution = pd.concat([
    target_distribution(y_train, "Train"),
    target_distribution(y_val, "Validation")
], ignore_index=True)

display(distribution)

,ensemble,classe,effectif,pourcentage
0,Train,0,38649,58.3496
1,Train,1,27588,41.6504
2,Validation,0,5141,55.0310
3,Validation,1,4201,44.9690


### 7. Baseline naïve

Avant d’entraîner un modèle complexe, on établit une référence minimale.

La baseline `most_frequent` prédit toujours la classe la plus fréquente observée dans le **Train**.

Un modèle ML utile devra apporter une valeur supérieure à cette référence, notamment sur les métriques adaptées à la classe `1`.


In [22]:
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)

y_val_pred_baseline = baseline.predict(X_val)

baseline_metrics = pd.DataFrame({
    "metrique": ["Accuracy", "Precision classe 1", "Recall classe 1", "F1 classe 1"],
    "validation": [
        accuracy_score(y_val, y_val_pred_baseline),
        precision_score(y_val, y_val_pred_baseline, zero_division=0),
        recall_score(y_val, y_val_pred_baseline, zero_division=0),
        f1_score(y_val, y_val_pred_baseline, zero_division=0),
    ]
})

display(baseline_metrics)

print("Matrice de confusion — baseline sur Validation")
display(pd.DataFrame(
    confusion_matrix(y_val, y_val_pred_baseline),
    index=["Réel 0", "Réel 1"],
    columns=["Prédit 0", "Prédit 1"]
))

,metrique,validation
0,Accuracy,0.5503
1,Precision classe 1,0.0000
2,Recall classe 1,0.0000
3,F1 classe 1,0.0000


Matrice de confusion — baseline sur Validation


,Prédit 0,Prédit 1
Réel 0,5141,0
Réel 1,4201,0


### Interprétation de la baseline

Une baseline de classe majoritaire peut avoir une accuracy apparemment correcte tout en obtenant :

- un **Recall classe 1 = 0** ;
- une **Precision classe 1 = 0** ;
- un **F1 classe 1 = 0**.

Cela illustre pourquoi l’**accuracy seule ne suffit pas** pour notre cas d’usage : notre objectif est précisément d’identifier les segments susceptibles de connaître une hausse critique.


### 8. Preprocessing sans fuite de données

Le preprocessing est appris **uniquement sur le Train** puis appliqué à la Validation. Les identifiants pays sont traités comme des catégories. Les valeurs catégorielles manquantes deviennent `INCONNU`; les numériques sont imputées par la médiane du Train avec indicateur de manque. Les catégories sont encodées en One-Hot et les numériques standardisées.

In [23]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

categorical_candidates = [
    "coo_id","coa_id","procedure_type","app_type",
    "dec_level","app_pc","origin_region","asylum_region"
]
categorical_features=[c for c in categorical_candidates if c in X_train.columns]
numeric_features=[c for c in X_train.columns if c not in categorical_features]

unexpected=[c for c in numeric_features if not pd.api.types.is_numeric_dtype(X_train[c])]
assert not unexpected, f"Colonnes non numériques dans le bloc numérique : {unexpected}"
assert all(c in categorical_features for c in ["coo_id","coa_id"] if c in X_train.columns)

print("Catégorielles :", categorical_features)
print("Numériques :", numeric_features)

Catégorielles : ['coo_id', 'coa_id', 'procedure_type', 'app_type', 'dec_level', 'app_pc', 'origin_region', 'asylum_region']
Numériques : ['year', 'applied', 'log_applied', 'applied_lag1', 'applied_lag2', 'absolute_change_past_1', 'absolute_change_past_2', 'growth_past_1', 'growth_past_2', 'two_consecutive_increases', 'applied_mean_3y', 'applied_std_3y', 'origin_applied_total_t', 'asylum_applied_total_t', 'log_origin_applied_total_t', 'log_asylum_applied_total_t', 'segment_share_origin_t', 'segment_share_asylum_t', 'origin_applied_total_lag1', 'idmc_total_origin_lag1', 'log_idmc_total_origin_lag1', 'idmc_pressure_ratio_lag1', 'decisions_total_lag1', 'protection_rate_lag1', 'rejection_rate_lag1']


### 8.1 Pipelines de transformation

`SimpleImputer` et `StandardScaler` n'apprennent leurs paramètres que pendant `.fit(X_train, y_train)`. `handle_unknown="ignore"` permet de traiter une catégorie nouvelle en Validation sans apprendre sur celle-ci.

In [24]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="INCONNU")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

print("✓ Preprocessor construit.")

✓ Preprocessor construit.


### 9. Régression Logistique

Premier vrai modèle ML : une Régression Logistique interprétable. Le preprocessing et le modèle sont réunis dans un même `Pipeline`, ce qui constitue un garde-fou contre les fuites entre Train et Validation.

In [25]:
logreg_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=2000, solver="liblinear", random_state=42))
])

logreg_pipeline.fit(X_train, y_train)
print("✓ Modèle entraîné uniquement sur le Train.")

✓ Modèle entraîné uniquement sur le Train.


### 10. Évaluation sur la Validation 2022–2023

Le seuil initial reste à **0,50**. Nous suivons Accuracy, Precision, Recall, F1, PR-AUC et ROC-AUC. Le Test 2024 → 2025 reste verrouillé.

In [26]:
y_val_proba_logreg = logreg_pipeline.predict_proba(X_val)[:,1]
y_val_pred_logreg = (y_val_proba_logreg >= 0.50).astype(int)

logreg_metrics = pd.DataFrame({
    "metrique":["Accuracy","Precision classe 1","Recall classe 1","F1 classe 1","PR-AUC","ROC-AUC"],
    "validation":[
        accuracy_score(y_val,y_val_pred_logreg),
        precision_score(y_val,y_val_pred_logreg,zero_division=0),
        recall_score(y_val,y_val_pred_logreg,zero_division=0),
        f1_score(y_val,y_val_pred_logreg,zero_division=0),
        average_precision_score(y_val,y_val_proba_logreg),
        roc_auc_score(y_val,y_val_proba_logreg)
    ]
})
display(logreg_metrics)
display(pd.DataFrame(confusion_matrix(y_val,y_val_pred_logreg),
                     index=["Réel 0","Réel 1"],columns=["Prédit 0","Prédit 1"]))

,metrique,validation
0,Accuracy,0.5622
1,Precision classe 1,0.5212
2,Recall classe 1,0.3247
3,F1 classe 1,0.4001
4,PR-AUC,0.4966
5,ROC-AUC,0.5625


,Prédit 0,Prédit 1
Réel 0,3888,1253
Réel 1,2837,1364


### 11. Comparaison avec la baseline

Le modèle doit apporter une valeur supérieure à la stratégie naïve. L'Accuracy seule ne suffit pas : le Recall mesure les hausses critiques réellement détectées et la Precision mesure la fiabilité des alertes.

In [27]:
comparison=pd.DataFrame({
    "Modèle":["Baseline majoritaire","Régression Logistique"],
    "Accuracy":[accuracy_score(y_val,y_val_pred_baseline),accuracy_score(y_val,y_val_pred_logreg)],
    "Precision_1":[precision_score(y_val,y_val_pred_baseline,zero_division=0),precision_score(y_val,y_val_pred_logreg,zero_division=0)],
    "Recall_1":[recall_score(y_val,y_val_pred_baseline,zero_division=0),recall_score(y_val,y_val_pred_logreg,zero_division=0)],
    "F1_1":[f1_score(y_val,y_val_pred_baseline,zero_division=0),f1_score(y_val,y_val_pred_logreg,zero_division=0)]
})
display(comparison)

,Modèle,Accuracy,Precision_1,Recall_1,F1_1
0,Baseline majoritaire,0.5503,0.0000,0.0000,0.0000
1,Régression Logistique,0.5622,0.5212,0.3247,0.4001


### 12. Contrôle du Feature Space et verrouillage du Test

On contrôle le nombre de variables après preprocessing. **Aucune métrique n'est calculée sur le Test 2024 → 2025 à ce stade.** Il ne sera ouvert qu'après choix du modèle et du seuil sur la Validation.

In [28]:
feature_names=logreg_pipeline.named_steps["preprocessor"].get_feature_names_out()
print("Features avant preprocessing :",X_train.shape[1])
print("Features après preprocessing :",len(feature_names))
print(feature_names[:30])

Features avant preprocessing : 33
Features après preprocessing : 470
['num__year' 'num__applied' 'num__log_applied' 'num__applied_lag1'
 'num__applied_lag2' 'num__absolute_change_past_1'
 'num__absolute_change_past_2' 'num__growth_past_1' 'num__growth_past_2'
 'num__two_consecutive_increases' 'num__applied_mean_3y'
 'num__applied_std_3y' 'num__origin_applied_total_t'
 'num__asylum_applied_total_t' 'num__log_origin_applied_total_t'
 'num__log_asylum_applied_total_t' 'num__segment_share_origin_t'
 'num__segment_share_asylum_t' 'num__origin_applied_total_lag1'
 'num__idmc_total_origin_lag1' 'num__log_idmc_total_origin_lag1'
 'num__idmc_pressure_ratio_lag1' 'num__decisions_total_lag1'
 'num__protection_rate_lag1' 'num__rejection_rate_lag1'
 'num__missingindicator_applied_lag1' 'num__missingindicator_applied_lag2'
 'num__missingindicator_absolute_change_past_1'
 'num__missingindicator_absolute_change_past_2'
 'num__missingindicator_growth_past_1']
